# Claude API 프롬프트 캐싱 (Prompt Caching)

큰 문서나 시스템 프롬프트를 반복 사용할 때, 캐싱으로 비용을 최대 ~90% 절감하고 응답 속도를 높이는 예제입니다.

**핵심 원리 — 캐싱은 접두사(prefix) 매칭이다**
- 렌더링 순서는 `tools` → `system` → `messages`
- `cache_control` 지점까지의 바이트가 정확히 같아야 캐시 히트
- 접두사 중간에 한 글자라도 바뀌면(타임스탬프 등) 그 뒤 전체가 무효화

**비용 구조**
| 구분 | 비용 |
|---|---|
| 캐시 쓰기 (최초 1회) | 기본 입력 단가의 ~1.25배 |
| 캐시 읽기 (이후) | 기본 입력 단가의 ~0.1배 |
| TTL | 기본 5분 (`ttl: "1h"`로 1시간 가능, 쓰기 2배) |

**주의**: 모델별 최소 캐싱 크기가 있습니다. `claude-opus-4-8`은 **4096 토큰 이상**이어야 캐싱됩니다.
그보다 짧으면 에러 없이 조용히 캐싱되지 않습니다.

## 1. 클라이언트 초기화

In [1]:
import os
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), ".env 파일에 ANTHROPIC_API_KEY를 설정해주세요"

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"
print("클라이언트 준비 완료")

클라이언트 준비 완료


## 2. 캐싱할 대용량 문서 준비

실무라면 사내 규정집, 제품 매뉴얼, 코드베이스 요약 같은 문서가 되겠지만,
여기서는 가상의 "클라우드 서비스 운영 매뉴얼"을 합성해서 만듭니다.
최소 캐싱 크기(4096 토큰)를 확실히 넘기도록 40개 장으로 구성합니다.

In [2]:
TOPICS = [
    "가상 머신 프로비저닝", "오토스케일링 정책", "로드밸런서 구성", "방화벽 규칙 관리",
    "객체 스토리지 수명주기", "블록 스토리지 스냅샷", "데이터베이스 백업", "읽기 전용 복제본",
    "VPC 네트워크 설계", "서브넷 분리 전략", "NAT 게이트웨이 운영", "프라이빗 엔드포인트",
    "IAM 역할 설계", "최소 권한 원칙", "감사 로그 보존", "비밀 키 로테이션",
    "모니터링 지표 수집", "알림 임계값 설정", "장애 대응 절차", "온콜 에스컬레이션",
    "배포 파이프라인", "블루그린 배포", "카나리 릴리스", "롤백 전략",
    "컨테이너 레지스트리", "쿠버네티스 노드풀", "파드 리소스 제한", "헬름 차트 관리",
    "비용 최적화 리뷰", "예약 인스턴스 정책", "미사용 리소스 정리", "태깅 표준",
    "재해 복구 계획", "다중 리전 이중화", "RTO/RPO 목표", "복구 훈련 절차",
    "규정 준수 점검", "데이터 암호화 표준", "접근 기록 검토", "보안 패치 주기",
]

sections = []
for i, topic in enumerate(TOPICS, start=1):
    sections.append(
        f"## 제{i}장 {topic}\n"
        f"{topic} 업무는 운영팀의 핵심 책임 영역이다. 담당자는 매 분기 초에 {topic} 관련 설정을 전수 점검하고, "
        f"변경이 필요한 항목은 변경관리위원회(CAB)의 승인을 받아 계획된 유지보수 window에만 적용한다. "
        f"{topic}의 표준 절차는 다음과 같다. 첫째, 현재 상태를 스냅샷으로 기록하고 변경 전후 비교가 가능하도록 한다. "
        f"둘째, 스테이징 환경에서 동일 변경을 먼저 적용하여 최소 24시간 관찰한다. "
        f"셋째, 운영 반영 시에는 반드시 2인 이상이 참여하는 4-eyes 원칙을 지키고, 작업 로그를 티켓 번호와 함께 남긴다. "
        f"넷째, 반영 후 30분간 핵심 지표(오류율, 지연시간, 포화도)를 모니터링하며 이상 징후 발견 시 즉시 롤백한다. "
        f"{topic} 관련 긴급 장애가 발생하면 P1 등급으로 분류하여 15분 이내 초동 대응을 시작해야 하며, "
        f"원인 분석 보고서(RCA)는 영업일 기준 5일 이내에 제출한다. 담당 부서는 인프라운영팀이며, "
        f"예외 승인 권한은 인프라총괄(Director) 이상에게 있다. 위반 시 내부 감사 대상이 된다."
    )

LARGE_DOC = "# 클라우드 서비스 운영 매뉴얼 v3.2\n\n" + "\n\n".join(sections)
print(f"문서 길이: {len(LARGE_DOC):,} 글자")

문서 길이: 22,273 글자


## 3. 토큰 수 확인 — 최소 캐싱 크기(4096) 넘는지 검증

`count_tokens` 엔드포인트로 실제 요청을 보내기 전에 토큰 수를 확인합니다.

In [3]:
SYSTEM_BLOCKS = [
    {
        "type": "text",
        "text": "당신은 사내 클라우드 운영 매뉴얼 전문 어시스턴트입니다. 아래 매뉴얼에 근거해서만 답변하세요.\n\n"
        + LARGE_DOC,
        "cache_control": {"type": "ephemeral"},  # 이 블록까지가 캐싱 대상 (기본 TTL 5분)
    }
]

count = client.messages.count_tokens(
    model=MODEL,
    system=SYSTEM_BLOCKS,
    messages=[{"role": "user", "content": "확인용"}],
)

print(f"전체 입력 토큰: {count.input_tokens:,}")
assert count.input_tokens >= 4096, "claude-opus-4-8 최소 캐싱 크기(4096 토큰)에 미달합니다 — 문서를 더 키우세요"
print("최소 캐싱 크기 통과 ✓")

전체 입력 토큰: 22,522
최소 캐싱 크기 통과 ✓


## 4. 첫 번째 요청 — 캐시 쓰기 (cache write)

처음 보내는 접두사이므로 `cache_creation_input_tokens`에 기록됩니다 (~1.25배 비용).

In [4]:
def ask(question: str):
    """동일한 system 접두사로 질문을 보내고 (응답, 소요시간)을 반환"""
    start = time.perf_counter()
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=SYSTEM_BLOCKS,  # 매 요청 바이트 단위로 동일해야 캐시 히트
        messages=[{"role": "user", "content": question}],
    )
    elapsed = time.perf_counter() - start
    return response, elapsed


def print_usage(response, elapsed):
    u = response.usage
    print(f"  소요 시간            : {elapsed:.1f}초")
    print(f"  캐시 쓰기 (1.25x 비용): {u.cache_creation_input_tokens:,} 토큰")
    print(f"  캐시 읽기 (0.1x 비용) : {u.cache_read_input_tokens:,} 토큰")
    print(f"  일반 입력 (1x 비용)   : {u.input_tokens:,} 토큰")


response1, elapsed1 = ask("P1 등급 장애의 초동 대응 시간과 RCA 제출 기한은?")

print("[답변]")
print(next(b.text for b in response1.content if b.type == "text"))
print("\n[사용량 — 첫 요청: 캐시 쓰기 발생]")
print_usage(response1, elapsed1)

[답변]
매뉴얼에 근거하여 답변드립니다.

- **초동 대응 시간**: P1 등급으로 분류된 긴급 장애는 **15분 이내**에 초동 대응을 시작해야 합니다.
- **RCA(원인 분석 보고서) 제출 기한**: **영업일 기준 5일 이내**에 제출해야 합니다.

이 기준은 매뉴얼 전 장(제1장~제40장)에 공통으로 적용되는 사항입니다.

[사용량 — 첫 요청: 캐시 쓰기 발생]
  소요 시간            : 4.0초
  캐시 쓰기 (1.25x 비용): 22,515 토큰
  캐시 읽기 (0.1x 비용) : 0 토큰
  일반 입력 (1x 비용)   : 35 토큰


## 5. 두 번째 요청 — 캐시 읽기 (cache read)

질문만 다르고 접두사(system)는 동일하므로, 이번엔 `cache_read_input_tokens`에 기록됩니다.
매뉴얼 전체가 ~0.1배 비용으로 처리되고 응답도 빨라집니다.

In [6]:
response2, elapsed2 = ask("운영 반영 시 지켜야 하는 4-eyes 원칙이 뭐야?")

print("[답변]")
print(next(b.text for b in response2.content if b.type == "text"))
print("\n[사용량 — 두 번째 요청: 캐시 히트]")
print_usage(response2, elapsed2)

if response2.usage.cache_read_input_tokens > 0:
    print("\n캐시 히트 성공 ✓ — 매뉴얼 부분이 ~0.1배 비용으로 처리됨")
else:
    print("\n캐시 미스 — TTL(5분) 초과 또는 접두사가 바뀌었는지 확인하세요")

[답변]
4-eyes 원칙은 **운영 반영 시 반드시 2인 이상이 참여**하도록 하는 원칙입니다.

매뉴얼의 표준 절차 세 번째 항목에 명시되어 있으며, 이때 **작업 로그를 티켓 번호와 함께 남겨야** 합니다.

즉, 운영 환경에 변경을 반영할 때 한 사람이 단독으로 작업하지 않고 최소 2명이 함께 참여하여 상호 확인하도록 하는 통제 장치입니다.

[사용량 — 두 번째 요청: 캐시 히트]
  소요 시간            : 4.2초
  캐시 쓰기 (1.25x 비용): 0 토큰
  캐시 읽기 (0.1x 비용) : 22,515 토큰
  일반 입력 (1x 비용)   : 36 토큰

캐시 히트 성공 ✓ — 매뉴얼 부분이 ~0.1배 비용으로 처리됨


## 6. 절감 효과 계산

두 요청의 사용량으로 캐싱이 없었을 때와 비용을 비교해봅니다.
(claude-opus-4-8 입력 단가 $5/1M 토큰 기준)

In [7]:
PRICE_PER_TOKEN = 5.00 / 1_000_000  # 입력 단가


def input_cost(usage):
    return (
        usage.cache_creation_input_tokens * PRICE_PER_TOKEN * 1.25
        + usage.cache_read_input_tokens * PRICE_PER_TOKEN * 0.10
        + usage.input_tokens * PRICE_PER_TOKEN
    )


def total_input_tokens(usage):
    return usage.cache_creation_input_tokens + usage.cache_read_input_tokens + usage.input_tokens


actual = input_cost(response1.usage) + input_cost(response2.usage)
without_cache = (total_input_tokens(response1.usage) + total_input_tokens(response2.usage)) * PRICE_PER_TOKEN

print(f"캐싱 사용 입력 비용   : ${actual:.6f}")
print(f"캐싱 미사용 입력 비용 : ${without_cache:.6f}")
print(f"절감률               : {(1 - actual / without_cache) * 100:.1f}%")
print("\n※ 요청이 반복될수록 절감 폭이 커집니다 — 쓰기는 1회, 읽기는 매번 0.1배")
print(f"※ 응답 속도: 1차 {elapsed1:.1f}초 → 2차 {elapsed2:.1f}초")

캐싱 사용 입력 비용   : $0.152331
캐싱 미사용 입력 비용 : $0.225505
절감률               : 32.4%

※ 요청이 반복될수록 절감 폭이 커집니다 — 쓰기는 1회, 읽기는 매번 0.1배
※ 응답 속도: 1차 4.0초 → 2차 4.2초


## 7. 실무 팁 — 캐시를 조용히 깨뜨리는 것들

캐시 히트가 안 나오면 (`cache_read_input_tokens`가 계속 0이면) 아래를 의심하세요.

| 패턴 | 왜 깨지나 |
|---|---|
| 시스템 프롬프트에 `datetime.now()` 삽입 | 매 요청 접두사 바이트가 달라짐 |
| UUID/요청 ID를 접두사 앞부분에 포함 | 마찬가지 |
| `json.dumps()`를 `sort_keys=True` 없이 사용 | 직렬화 순서가 비결정적 |
| 요청마다 tools 목록이 바뀜/순서 변경 | tools는 접두사 맨 앞에 렌더링됨 |
| 모델을 중간에 변경 | 캐시는 모델별로 분리됨 |

**해결 원칙**: 안정적인 내용(고정 시스템 프롬프트, 툴 목록)은 앞에, 매번 바뀌는 내용(질문, 타임스탬프)은
마지막 `cache_control` 지점 **뒤에** 배치하세요.

**멀티턴 대화 캐싱**: 대화가 길어질 때는 요청 최상위에 `cache_control={"type": "ephemeral"}`을 주면
마지막 캐싱 가능 블록에 자동으로 캐시 지점이 잡혀서, 턴이 쌓일수록 이전 대화 전체가 캐시에서 읽힙니다.

In [ ]:
# 멀티턴 자동 캐싱 예시 — 최상위 cache_control로 마지막 캐싱 가능 블록에 자동 배치
history = [{"role": "user", "content": "카나리 릴리스 절차를 요약해줘."}]

r1 = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    cache_control={"type": "ephemeral"},  # 자동으로 마지막 캐싱 가능 블록에 적용
    system=SYSTEM_BLOCKS,
    messages=history,
)
history.append({"role": "assistant", "content": r1.content})
history.append({"role": "user", "content": "그럼 롤백은 언제 해야 해?"})

r2 = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    cache_control={"type": "ephemeral"},
    system=SYSTEM_BLOCKS,
    messages=history,
)

print("[2번째 턴 답변]")
print(next(b.text for b in r2.content if b.type == "text"))
print("\n[사용량]")
u = r2.usage
print(f"  캐시 읽기: {u.cache_read_input_tokens:,} / 캐시 쓰기: {u.cache_creation_input_tokens:,} / 일반 입력: {u.input_tokens:,}")
print("→ 시스템 프롬프트 + 1턴째 대화가 캐시에서 읽히고, 새로 추가된 부분만 쓰기/일반 처리됩니다")